# Day 15 - Filter Engine Core

## Objective

- Load validated financial ratio data
- Build a reusable company filter engine
- Support multiple financial conditions
- Apply AND logic between filters
- Generate screener results
- Export filtered company data

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df = pd.read_csv("../reports/financial_ratios_day12.csv")

print("Financial Ratio Data Loaded Successfully")
print("Shape:", df.shape)

df.head()

Financial Ratio Data Loaded Successfully
Shape: (1184, 7)


,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio
0,ABB,Dec 2012,22.411128,0.0,NaN,1.822492,17.241379
1,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
2,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
3,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755
4,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755


In [4]:
print("Available Columns:")

for col in df.columns:
    print(col)

Available Columns:
company_id
year
return_on_equity_pct
debt_to_equity
interest_coverage
asset_turnover
dividend_payout_ratio


In [5]:
numeric_cols = [
    "return_on_equity_pct",
    "debt_to_equity",
    "interest_coverage",
    "asset_turnover",
    "dividend_payout_ratio"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print(df[numeric_cols].dtypes)

return_on_equity_pct     float64
debt_to_equity           float64
interest_coverage        float64
asset_turnover           float64
dividend_payout_ratio    float64
dtype: object


In [6]:
def filter_companies(
    data,
    roe_min=None,
    debt_to_equity_max=None,
    interest_coverage_min=None,
    asset_turnover_min=None,
    dividend_payout_min=None
):
    result = data.copy()

    if roe_min is not None:
        result = result[
            result["return_on_equity_pct"] >= roe_min
        ]

    if debt_to_equity_max is not None:
        result = result[
            result["debt_to_equity"] <= debt_to_equity_max
        ]

    if interest_coverage_min is not None:
        result = result[
            result["interest_coverage"] >= interest_coverage_min
        ]

    if asset_turnover_min is not None:
        result = result[
            result["asset_turnover"] >= asset_turnover_min
        ]

    if dividend_payout_min is not None:
        result = result[
            result["dividend_payout_ratio"] >= dividend_payout_min
        ]

    return result

In [7]:
roe_filter = filter_companies(
    df,
    roe_min=15
)

print("Companies passing ROE filter:", len(roe_filter))

roe_filter.head()

Companies passing ROE filter: 672


,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio
0,ABB,Dec 2012,22.411128,0.0,NaN,1.822492,17.241379
1,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
2,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
3,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755
4,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755


In [8]:
debt_filter = filter_companies(
    df,
    debt_to_equity_max=1
)

print(
    "Companies passing Debt-to-Equity filter:",
    len(debt_filter)
)

debt_filter.head()

Companies passing Debt-to-Equity filter: 700


,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio
0,ABB,Dec 2012,22.411128,0.0,NaN,1.822492,17.241379
1,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
2,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
3,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755
4,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755


In [9]:
quality_filter = filter_companies(
    df,
    roe_min=15,
    debt_to_equity_max=1,
    interest_coverage_min=3
)

print(
    "Companies passing all quality filters:",
    len(quality_filter)
)

quality_filter.head(20)

Companies passing all quality filters: 420


,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio
5,ABB,Mar 2016,21.338912,0.000000,121.666667,1.617574,11.372549
6,ABB,Mar 2016,21.338912,0.000000,121.666667,1.617574,11.372549
7,ABB,Mar 2017,19.971161,0.000000,199.000000,1.405131,11.191336
8,ABB,Mar 2017,19.971161,0.000000,199.000000,1.405131,11.191336
9,ABB,Mar 2018,23.685765,0.000000,131.250000,1.365066,7.231920
10,ABB,Mar 2018,23.685765,0.000000,131.250000,1.365066,7.231920
11,ABB,Mar 2019,22.410359,0.000000,302.500000,1.250935,6.888889
12,ABB,Mar 2019,22.410359,0.000000,302.500000,1.250935,6.888889
13,ABB,Mar 2020,24.393254,0.071987,84.111111,1.153933,15.177066
14,ABB,Mar 2020,24.393254,0.071987,84.111111,1.153933,15.177066


In [10]:
advanced_filter = filter_companies(
    df,
    roe_min=15,
    debt_to_equity_max=1,
    interest_coverage_min=3,
    asset_turnover_min=0.5
)

print(
    "Companies passing advanced filter:",
    len(advanced_filter)
)

advanced_filter.head(20)

Companies passing advanced filter: 381


,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio
5,ABB,Mar 2016,21.338912,0.000000,121.666667,1.617574,11.372549
6,ABB,Mar 2016,21.338912,0.000000,121.666667,1.617574,11.372549
7,ABB,Mar 2017,19.971161,0.000000,199.000000,1.405131,11.191336
8,ABB,Mar 2017,19.971161,0.000000,199.000000,1.405131,11.191336
9,ABB,Mar 2018,23.685765,0.000000,131.250000,1.365066,7.231920
10,ABB,Mar 2018,23.685765,0.000000,131.250000,1.365066,7.231920
11,ABB,Mar 2019,22.410359,0.000000,302.500000,1.250935,6.888889
12,ABB,Mar 2019,22.410359,0.000000,302.500000,1.250935,6.888889
13,ABB,Mar 2020,24.393254,0.071987,84.111111,1.153933,15.177066
14,ABB,Mar 2020,24.393254,0.071987,84.111111,1.153933,15.177066


In [11]:
filter_summary = pd.DataFrame({
    "filter_name": [
        "ROE >= 15%",
        "Debt to Equity <= 1",
        "Quality Filter",
        "Advanced Filter"
    ],
    "records_found": [
        len(roe_filter),
        len(debt_filter),
        len(quality_filter),
        len(advanced_filter)
    ]
})

filter_summary

,filter_name,records_found
0,ROE >= 15%,672
1,Debt to Equity <= 1,700
2,Quality Filter,420
3,Advanced Filter,381


In [12]:
quality_filter.to_csv(
    "../reports/day15_screener_output.csv",
    index=False
)

filter_summary.to_csv(
    "../reports/day15_filter_summary.csv",
    index=False
)

print("Day 15 Screener Reports Saved Successfully!")

Day 15 Screener Reports Saved Successfully!


# Day 15 Summary

## Filter Engine Core Completed

- Loaded validated financial ratio data.
- Implemented a reusable company filter engine.
- Added ROE filtering.
- Added Debt-to-Equity filtering.
- Added Interest Coverage filtering.
- Added Asset Turnover filtering.
- Added Dividend Payout filtering.
- Implemented multiple filter AND logic.
- Generated screener output and filter summary reports.
- Created a reusable screener engine module.

## Sprint 3 Status

The core filter engine is ready for preset screeners and advanced screener configurations.